In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import json
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
from pandas import Series, DataFrame
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os
import argparse
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold # 导入用于内部验证集划分
import torch.nn as nn
import numpy as np
import random
import logging
import sys
from monai.data import CacheDataset,DataLoader
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, classification_report, confusion_matrix, accuracy_score, roc_auc_score
import scipy.stats as stats
from tqdm import tqdm 
import csv
import torchvision.transforms.functional as F

In [ ]:
def read_split_data_by_hospital(
    csv_file_path: str,
    rad_file_path: str, # 新增：放射组学特征文件路径
    image_root: str,
    train_hospitals: list = [3, 5],
    random_seed: int = 0
):
    random.seed(random_seed)
    np.random.seed(random_seed)

    assert os.path.exists(csv_file_path), f"CSV file: {csv_file_path} does not exist."
    assert os.path.exists(rad_file_path), f"Radiomics CSV file: {rad_file_path} does not exist." # 检查放射组学文件
    assert os.path.exists(image_root), f"Image root directory: {image_root} does not exist."

    # --- 1. 加载并初步处理原始临床数据 ---
    df_raw = pd.read_csv(csv_file_path, encoding='gbk')
    df_raw['patient_ID'] = df_raw['patient_ID'].astype(str).str.strip()
    df_raw['bpCR'] = df_raw['bpCR'].astype(int)
    df_raw['hospital'] = df_raw['hospital'].astype(int)

    # --- 2. 加载并标准化放射组学数据 ---
    df_rad = pd.read_csv(rad_file_path, encoding='gbk')
    df_rad['patient_ID'] = df_rad['patient_ID'].astype(str).str.strip()
    # 确保 hospital 列在放射组学数据中也是正确的类型，以便后续合并
    if 'hospital' in df_rad.columns:
        df_rad['hospital'] = df_rad['hospital'].astype(int)
    else:
        # 如果放射组学文件不包含hospital列，可能需要从df_raw中补充或跳过合并时的hospital列
        raise ValueError("Radiomics CSV file must contain a 'hospital' column for merging.")

    # 提取放射组学特征列 (与他人代码保持一致)
    rad_cols = [col for col in df_rad.columns 
                       if any(x in col for x in ['preDCE_', 'preDWI_'])]
    
    # 初始化 StandardScaler 并进行 Z-score 标准化
    scaler = StandardScaler()
    df_rad[rad_cols] = scaler.fit_transform(df_rad[rad_cols])

    # --- 3. 合并临床数据和放射组学特征 ---
    # 使用 patient_ID 和 hospital 进行合并，以确保唯一性
    df_merged = pd.merge(
        df_raw, 
        df_rad[['patient_ID', 'hospital'] + rad_cols], 
        on=['patient_ID', 'hospital'], 
        how='left' # 保留所有临床数据，即使无放射组学特征
    )

    # --- 4. 定义临床特征列并初始化预处理器 ---
    clinical_cols = ['T_stage', 'HER2_status', 'NAC_classification', 'ER_status', 'PR_status', 'Ki_67', 'age', 'age_menarche', 'BMI',
                                 'treatment_duration', 'NAC_treatment_cycle', 'endocrine', 'Histological_type', 'result_ipsilateral_axillary_LNP', 
                                 'N_stage', 'M_stage', 'enlargement_unilateral_axillary_LN', 'enlargement_unilateral_clavicular_LN', 'enlargement_Ipsilateral_internal_mammary_LN',
                                 'target_lesion_location', 'target_lesion_quadrant']
    
    # 定义特征分类（与他人代码保持一致）
    numeric_features = ['Ki_67', 'age', 'age_menarche', 'BMI']
    ordinal_features = ['HER2_status', 'T_stage', 'N_stage', 'M_stage']
    nominal_features = ['PR_status', 'ER_status']
    raw_features = ['NAC_classification', 'treatment_duration', 'NAC_treatment_cycle', 'endocrine', 'Histological_type', 'result_ipsilateral_axillary_LNP',
                                 'enlargement_unilateral_axillary_LN', 'enlargement_unilateral_clavicular_LN', 'enlargement_Ipsilateral_internal_mammary_LN',
                                 'target_lesion_location', 'target_lesion_quadrant']

    # 初始化临床数据预处理器
    preprocessor = ColumnTransformer(transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')), 
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)) # 确保处理未知类别
        ]), ordinal_features),
        ('nominal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('encoder', OneHotEncoder(handle_unknown='ignore')) # 确保处理未知类别
        ]), nominal_features),
        ('raw', 'passthrough', raw_features)
    ])
    
    # 这里我们使用 df_merged 中的临床列进行 fit，以确保编码器能学习到所有可能的值。
    preprocessor.fit(df_merged[clinical_cols])

    # --- 5. 构建患者信息列表 (包含图像路径、标签、医院ID、处理后的临床特征和放射组学特征) ---
    all_patient_info = [] 
    class_labels = sorted(df_merged['bpCR'].unique().tolist())
    class_indices = dict((k, v) for v, k in enumerate(class_labels))
    print(f"Class indices: {json.dumps(dict((val, key) for key, val in class_indices.items()), indent=4)}")
    

    for idx, row in df_merged.iterrows(): # 遍历合并后的DataFrame
        patient_id = str(row['patient_ID'])
        hospital_id = int(row['hospital'])
        label = int(row['bpCR'])

        pre_dwi_path = os.path.join(image_root, f"{patient_id}_dwi1.png") # 注意，这里是png，不是nii.gz
        pre_dce_path = os.path.join(image_root, f"{patient_id}_c1.png") # 注意，这里是png，不是nii.gz
        
        # 检查图像文件是否存在
        if not (os.path.exists(pre_dwi_path) and os.path.exists(pre_dce_path)):
            missing_images = []
            if not os.path.exists(pre_dwi_path): missing_images.append(f"{patient_id}_dwi1.png")
            if not os.path.exists(pre_dce_path): missing_images.append(f"{patient_id}_c1.png")
            print(f"Warning: Missing one or more MRI sequences for patient {patient_id}. Skipping patient. Missing files: {', '.join(missing_images)}")
            continue

        # 预处理临床数据
        # preprocessor.transform 需要一个 DataFrame 或 2D 数组
        clinical_data_single_row = row[clinical_cols].to_frame().T 
        processed_clinical = preprocessor.transform(clinical_data_single_row)
        
        # 处理稀疏矩阵（OneHotEncoder 可能生成）并展平
        if hasattr(processed_clinical, 'toarray'):
            processed_clinical = processed_clinical.toarray().flatten()
        else:
            processed_clinical = processed_clinical.flatten()
        
        # 获取放射组学特征
        # 确保这些列在 df_merged 中存在，并且已经标准化
        radiomics_features = row[rad_cols].values.astype(np.float32)

        all_patient_info.append({
            'patient_id': patient_id,
            'label': label,
            'hospital': hospital_id,
            'pre_dwi': pre_dwi_path,
            'pre_dce': pre_dce_path,
            'clinical': processed_clinical.astype(np.float32), # 添加处理后的临床特征
            'radiomics': radiomics_features.astype(np.float32) # 添加标准化后的放射组学特征
        })

    if not all_patient_info:
        raise ValueError("No complete patient data found based on the provided CSVs and image root.")

    all_patient_df_processed = pd.DataFrame(all_patient_info)

    # 划分训练集和外部测试集
    train_val_data = all_patient_df_processed[all_patient_df_processed['hospital'].isin(train_hospitals)].to_dict('records')
    external_test_data = all_patient_df_processed[~all_patient_df_processed['hospital'].isin(train_hospitals)]
    
    dict_of_test_sets = {}
    test_hospital_ids = sorted(external_test_data['hospital'].unique().tolist())
    
    for hospital_id in test_hospital_ids:
        hospital_data = external_test_data[external_test_data['hospital'] == hospital_id].to_dict('records')
        dict_of_test_sets[hospital_id] = hospital_data

    # 打印统计信息
    print(f"\n--- Dataset Split Statistics ---")
    print(f"Total complete patient MRI sequences with clinical/radiomics data found: {len(all_patient_info)}")
    print(f"Train/Validation pool patients: {len(train_val_data)} (Hospitals: {train_hospitals})")
    
    print("\n--- External Test Set Details ---")
    if not dict_of_test_sets:
        print("No external test sets defined.")
    else:
        for hospital_id, data_list in dict_of_test_sets.items():
            print(f"   Hospital {hospital_id}: {len(data_list)} patients")
            if data_list:
                labels = [d['label'] for d in data_list]
                # 这里假设 class_indices 仍然可以访问，或者可以从 df_merged 中重新计算
                print(f"     Class Distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")

    return train_val_data, dict_of_test_sets


class MyDataSet(Dataset):
    """
    一个用于加载 MRI 图像数据集的自定义数据集类。
    支持加载 DWI, CE-T1WI, T2WI 三种模态的图像，并返回相应的张量、标签、图像名称、临床特征和放射组学特征。
    """
    def __init__(self, patient_data_list, transform=None):
        self.patient_data_list = patient_data_list
        self.transform = transform

    def __len__(self):
        return len(self.patient_data_list)

    def __getitem__(self, item):
        patient_info = self.patient_data_list[item]

        pre_dwi_path = patient_info['pre_dwi']
        pre_dce_path = patient_info['pre_dce']
        label = patient_info['label']
        patient_id = patient_info['patient_id']
        # 新增：获取处理后的临床特征和放射组学特征
        clinical_features = patient_info['clinical']
        radiomics_features = patient_info['radiomics']

        # 加载图像并转换为灰度图
        pre_dwi = Image.open(pre_dwi_path).convert('L')
        pre_dce = Image.open(pre_dce_path).convert('L')

        # # 如果有transform，则对所有图像应用相同的transform
        # if self.transform is not None:
        #     pre_dwi = self.transform(pre_dwi)
        #     pre_dce = self.transform(pre_dce)

        if self.transform is not None:
            # 判断 transform 的类型来选择正确的处理方式
            if isinstance(self.transform, PairedTransforms):
                # 训练阶段：使用 PairedTransforms，对所有四张图像应用相同随机变换
                pre_dwi, pre_dce = self.transform(pre_dwi, pre_dce)
            elif isinstance(self.transform, transforms.Compose):
                # 测试阶段：使用 Compose，对每张图像分别应用确定性变换
                pre_dwi = self.transform(pre_dwi)
                pre_dce = self.transform(pre_dce)

        # 返回所有需要的数据
        return pre_dwi, pre_dce, \
               torch.from_numpy(clinical_features), \
               torch.from_numpy(radiomics_features), \
               label, patient_id

    @staticmethod
    def collate_fn(batch):
        """
        自定义 collate_fn，用于将多个样本组成一个批次。
        这个函数需要适应新的返回格式 (dwi_image, cet1wi_image, clinical_features, radiomics_features, label, patient_id)。
        """
        # 注意：这里需要与 __getitem__ 的返回顺序和数量严格对应
        pre_dwi, pre_dce, clinical_features, radiomics_features, labels, patient_ids = zip(*batch)

        # 将图像张量堆叠成批次
        pre_dwi_batch = torch.stack(pre_dwi, 0)
        pre_dce_batch = torch.stack(pre_dce, 0)

        # 将临床特征和放射组学特征堆叠成批次
        clinical_features_batch = torch.stack(clinical_features, 0)
        radiomics_features_batch = torch.stack(radiomics_features, 0)

        # 将标签转换为 Tensor
        labels_batch = torch.as_tensor(labels, dtype=torch.long)

        # 返回字典形式，方便在训练循环中解包
        return {
            'pre_dwi': pre_dwi_batch,
            'pre_dce': pre_dce_batch,
            'clinical': clinical_features_batch,
            'radiomics': radiomics_features_batch,
            'label': labels_batch,
            'patient_ids': patient_ids # patient_ids 通常不需要转换为 Tensor
        }


def get_train_transforms(img_size=224):
    """
    获取训练阶段的图像预处理变换，确保成对图像（pre/post）应用相同的随机变换。
    """
    # 训练阶段的随机变换列表
    train_transforms = [
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0), ratio=(3./4, 4./3)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=0)
    ]
    
    # 测试阶段的确定性变换
    # 这部分变换通常在数据增强后应用，以确保图像尺寸和归一化的一致性
    final_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    return PairedTransforms(train_transforms, final_transforms)


def get_test_transforms(img_size=224):
    """
    获取测试阶段的图像预处理变换，不包含随机性。
    """
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])


class PairedTransforms:
    def __init__(self, random_transforms, final_transforms):
        self.random_transforms = random_transforms
        self.final_transforms = final_transforms

    def __call__(self, img_pre_dwi, img_pre_dce):
        # 设置随机种子
        seed = random.randint(0, 100000)
        
        # 特殊处理 RandomResizedCrop
        transform = self.random_transforms[0]
        i, j, h, w = transform.get_params(img_pre_dwi, transform.scale, transform.ratio)
        img_pre_dwi = F.resized_crop(img_pre_dwi, i, j, h, w, transform.size, transform.interpolation)
        img_pre_dce = F.resized_crop(img_pre_dce, i, j, h, w, transform.size, transform.interpolation)


        # 循环处理其他随机变换
        for transform in self.random_transforms[1:]:
            torch.manual_seed(seed)
            img_pre_dwi = transform(img_pre_dwi)
            torch.manual_seed(seed)
            img_pre_dce = transform(img_pre_dce)

        # 应用确定性变换
        img_pre_dwi = self.final_transforms(img_pre_dwi)
        img_pre_dce = self.final_transforms(img_pre_dce)
        
        return img_pre_dwi, img_pre_dce


# 裁剪范围在10和5之间
train_data, test_data = read_split_data_by_hospital(
        csv_file_path='./dataset_csv/new_1292.csv',
        rad_file_path='./dataset_csv/radiomics_features_delta.csv',
        image_root='/data/data/FromHospital/YiZhong-HE-Neoadjuvant-Breast/Clean_Data/Again_croppng',
        train_hospitals=[3, 5], # 默认是 [3, 5]
        random_seed=42
    )


# 验证结果
print("训练集示例：")
# print(train_data[0])
print(f"\n训练集数量：{len(train_data)}，验证集数量：{len(test_data)}")

img_size = 224 # 确保 args 中有 img_size
train_preprocess = get_train_transforms(img_size)
val_preprocess = get_test_transforms(img_size)

In [ ]:
# 最终的模型

import torch
import torch.nn as nn
from model_second import FocalLoss, CMFA, ClinicalProcessor, RadiomicsProcessor, CrossModalAttention, DynamicFusion
from model_first import ResNetEncoder2d, CBAM2d, GELU
from models.Sptmodel import SpectralTransform


class GatedDynamicFusion(nn.Module):
    def __init__(self, in_channels):
        super(GatedDynamicFusion, self).__init__()
        # 定义门控机制
        # 使用一个卷积层来生成门控值，它将决定DCE和DWI特征的组合权重
        self.gate_conv = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, stride=1, padding=0),
            nn.Sigmoid()
        )
        
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=3, stride=1, padding=1),
            GELU()
        )
        
    def forward(self, dce_features, dwi_features):
        """
        Args:
            dce_features (torch.Tensor): DCE图像编码后的特征 [B, C, H, W]
            dwi_features (torch.Tensor): DWI图像编码后的特征 [B, C, H, W]
        """
        # 将DCE和DWI特征拼接在一起，作为门控和融合的输入
        combined_features = torch.cat([dce_features, dwi_features], dim=1) # [B, 2C, H, W]
        
        # 生成门控值。门控值是一个介于0和1之间的张量，
        # 它代表了DCE特征在每个空间位置上的重要性
        dce_gate = self.gate_conv(combined_features) # [B, C, H, W]
        
        # 计算DWI的门控值。DWI的重要性与DCE互补
        dwi_gate = 1 - dce_gate
        
        # 使用门控值对DCE和DWI特征进行动态加权
        dce_features_gated = dce_features * dce_gate
        dwi_features_gated = dwi_features * dwi_gate
        
        # 门控后的特征进行融合，让模型自己学习最佳的组合方式
        fused_features = self.fusion_conv(torch.cat([dce_features_gated, dwi_features_gated], dim=1))
        
        # # 或者更简单的加权融合
        # fused_features = dce_features * dce_gate + dwi_features * dwi_gate
        
        return fused_features


# 加入了spt
class MedicalImageProcessor(nn.Module):
    def __init__(self, in_channel=1,d_model=256, dropout=0.25,weights_path=None, device=torch.device("cuda")):
        super(MedicalImageProcessor, self).__init__()
        self.device = device
        self.weights_path = weights_path
        
        self.encoder_dce = ResNetEncoder2d(in_channels=in_channel)
        self.encoder_dwi = ResNetEncoder2d(in_channels=in_channel)

        if weights_path:
            self._load_and_freeze_weights()
        
        self.fusion_module = GatedDynamicFusion(in_channels=d_model)
        self.spectral_fusion = SpectralTransform(in_channels=d_model, out_channels=d_model)
        self.final_fusion_conv = nn.Sequential(
            nn.Conv2d(d_model * 2, d_model, kernel_size=1),
            GELU()
        )
        
        # 新增：全局平均池化层，用于将特征图展平为向量
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def _load_and_freeze_weights(self):
        if not os.path.exists(self.weights_path):
            print(f"警告：权重文件 {self.weights_path} 不存在。Encoder将使用随机初始化权重")
            return
        device = next(self.parameters()).device
        weights = torch.load(self.weights_path, map_location=device)

        dce_weights = {k.replace('encoder_dce.', ''): v for k, v in weights.items()
                       if k.startswith('encoder_dce.')}
        missing_dce, _ = self.encoder_dce.load_state_dict(dce_weights, strict=False)

        dwi_weights = {k.replace('encoder_dwi.', ''): v for k, v in weights.items()
                       if k.startswith('encoder_dwi.')}
        missing_dwi, _ = self.encoder_dwi.load_state_dict(dwi_weights, strict=False)

        # # 冻结 Encoder 权重 (如果需要)
        # for param in self.encoder_dce.parameters():
        #     param.requires_grad = False
        # for param in self.encoder_dwi.parameters():
        #     param.requires_grad = False
        # print("Encoder 权重已冻结.")

    def forward(self, pre_dce, pre_dwi):
        encoded_pre_dce = self.encoder_dce(pre_dce)
        encoded_pre_dwi = self.encoder_dwi(pre_dwi)
        fused_features = self.fusion_module(encoded_pre_dce, encoded_pre_dwi)

        # 这有助于模型同时利用空间和频谱信息
        fused_features_spt = self.spectral_fusion(encoded_pre_dce, encoded_pre_dwi)
        # 将空间特征与频谱特征相加，以实现信息融合
        combined_features = torch.cat([fused_features, fused_features_spt], dim=1)
        final_fused_features = self.final_fusion_conv(combined_features)
        
        # 全局池化，展平为向量
        token_img = self.global_pool(final_fused_features).squeeze()
        
        # 确保当批次大小为1时，token_img仍然是2维张量
        if token_img.dim() == 1:
            token_img = token_img.unsqueeze(0)
            
        return token_img
   

class GatedVectorFusion(nn.Module):
    def __init__(self, d_model):
        super(GatedVectorFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )
        
    def forward(self, img_token, tab_token):
        # 拼接图像和表格特征
        combined_features = torch.cat([img_token, tab_token], dim=-1) # [B, 2*d_model]
        # 计算门控值
        gate_weights = self.gate(combined_features) # [B, d_model]
        # 动态加权融合
        fused_token = img_token * gate_weights + tab_token * (1 - gate_weights)
        
        return fused_token


# 主融合模型
class DoubleTower(nn.Module):
    def __init__(self, in_channel=1, clinical_dim=23, rad_dim=2264, d_model=256, dropout_rate=0.25, num_classes=2, weights_path=None, focal_alpha=None, focal_gamma=2.0, device=torch.device("cuda")):
        super().__init__()
        self.device = device
        # 初始化各处理器
        # self.image_processor = MedicalImageProcessor_CMT(in_channel=in_channel,d_model=d_model, dropout=dropout_rate, weights_path=weights_path, device=device)
        self.image_processor = MedicalImageProcessor(in_channel=in_channel,d_model=d_model, dropout=dropout_rate,weights_path=weights_path, device=device)
        self.rad_processor = RadiomicsProcessor(input_dim=rad_dim, d_model=d_model, dropout=dropout_rate)
        self.clin_processor = ClinicalProcessor(input_dim=clinical_dim, d_model=d_model, dropout=dropout_rate)

        self.cross_attn = CrossModalAttention(d_model=d_model)
        self.cmfa = CMFA(img_dim=d_model, tab_dim=d_model, hid_dim=d_model, heads=4, dropout=dropout_rate)
        self.fusion = DynamicFusion(d_model=d_model)
        self.gated_fusion = GatedVectorFusion(d_model=d_model)
       
        # 分类器，增加非线性层数
        self.classifier = nn.Sequential(
            nn.Linear(d_model*2, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        ).to(device)

         # === 新增损失函数配置 ===
        self.focal_loss = FocalLoss(
            class_num=num_classes,
            alpha=focal_alpha,  # 建议设置为[1-pCR_rate, pCR_rate]
            gamma=focal_gamma
        )


    def forward(self,  x1, x2, x_clin, x_rad, label=None):
        # 处理各模态数据
        # 只用临床特征进行拼接得到的预测结果
        token_img = self.image_processor(x1, x2) # [B,256]
        token_rad = self.rad_processor(x_rad) # [B,256]
        token_clin = self.clin_processor(x_clin) # [B,256]

        # 增强的交互模块 —— 分类器的输入维度是 256
        attn_enhanced = self.cross_attn(token_clin, token_rad)
        token_rad_clin = self.fusion(token_clin, attn_enhanced) # [B,256]

        tokens = self.cmfa(token_img, token_rad_clin) # [B,512]

        # inal_fused_token = self.gated_fusion(token_img, token_rad_clin) # [B, d_model]

        # tokens = torch.concat([token_img, token_rad_clin], dim=-1)
        # tokens = self.cross_attn(token_rad_clin,token_img)

        logits = self.classifier(tokens)
        
        # === 多任务损失计算 ===
        if label is not None:
            # 1. Focal Loss
            fl = self.focal_loss(logits, label)
            
            # 动态加权总损失
            total_loss = fl
            
            return logits, total_loss
        return logits    


In [ ]:
# 五折交叉验证

from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from scipy import interp


# 设定交叉验证折数
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

weights_path = './weights/first_weight/best_model_fold2.pth'
# weights_path = None

# 确保输出目录存在
output_dir = './weights_bigcrop'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "multi"), exist_ok=True) # 专门用于存放mask的子目录
# os.makedirs(os.path.join(output_dir, "img_rad"), exist_ok=True)

output_png = './crossval_pngs'
os.makedirs(output_png, exist_ok=True)

# 提取训练开发数据和标签
X = np.arange(len(train_data))  # 用索引来划分
y = [item['label'] for item in train_data]

# 用于记录每一折的表现
fold_results = []
all_fpr = []      # 新增：存储每折的FPR
all_tpr = []      # 新增：存储每折的TPR
all_auc = []      # 新增：存储每折的AUC

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== Fold {fold+1}/{n_splits} ===")

    # 初始化当前fold的记录
    fold_train_losses = []
    fold_val_losses = []

    save_path = os.path.join(output_dir, "multi", f"best_model_fold{fold+1}.pth")

    # 获取该折的训练和验证数据
    train_fold = [train_data[i] for i in train_idx]
    val_fold = [train_data[i] for i in val_idx]

    # 构建 Dataset 和 DataLoader
    train_ds = MyDataSet(patient_data_list=train_fold, transform=train_preprocess)
    val_ds = MyDataSet(patient_data_list=val_fold, transform=val_preprocess)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, collate_fn=train_ds.collate_fn, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=16, num_workers=4, collate_fn=val_ds.collate_fn, pin_memory=True)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # 初始化模型、优化器
    model = DoubleTower(in_channel=1, dropout_rate=0.25, focal_alpha=[0.65, 0.35],focal_gamma=1.7, weights_path=weights_path,device=device).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)
    loss_function = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(device))

    val_interval = 1
    best_metric = -1
    best_metric_epoch = -1
    max_epochs = 60
    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        step = 0
        for batch_data in train_loader:
            step += 1
            pre_dce, pre_dwi, input_clinical, input_rad, labels = batch_data["pre_dce"].to(device), batch_data['pre_dwi'].to(device), batch_data["clinical"].to(device), batch_data["radiomics"].to(device), batch_data["label"].to(device)
            optimizer.zero_grad()
            outputs, conloss = model(pre_dce, pre_dwi, input_clinical, input_rad, labels)

            loss = loss_function(outputs, labels)
            loss = loss + conloss
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        epoch_loss /= step
        print(f"epoch {epoch + 1} average  train loss: {epoch_loss:.4f}")

        if (epoch + 1) % val_interval == 0:
            model.eval()
            y_pred_list = []
            y_true_list = []
            val_loss = 0
            step2 = 0
            
            with torch.no_grad():
                for val_data in val_loader:
                    step2 += 1
                    pre_dce, pre_dwi, input_clinical, input_rad, val_labels = val_data["pre_dce"].to(device), val_data['pre_dwi'].to(device), val_data["clinical"].to(device), val_data["radiomics"].to(device), val_data["label"].to(device)
                    val_output = model(pre_dce, pre_dwi, input_clinical, input_rad)
                    y_pred_list.append(val_output)
                    y_true_list.append(val_labels)

                    val_loss += loss_function(val_output, val_labels).item()

                val_loss /= step2
                y_pred = torch.cat(y_pred_list, dim=0)
                y_true = torch.cat(y_true_list, dim=0)
                y_prob = torch.softmax(y_pred, dim=1)
                y_score = y_prob[:, 1].detach().cpu().numpy()
                y_true_np = y_true.detach().cpu().numpy()

                auc_result = roc_auc_score(y_true_np, y_score)
                y_pred_label = np.argmax(y_prob.cpu().numpy(), axis=1)
                acc_metric = (y_pred_label == y_true_np).mean()

                
                if auc_result > best_metric:
                    best_metric = auc_result
                    best_metric_epoch = epoch + 1
                    torch.save(model.state_dict(), save_path)
                    print("Saved new best model for this fold.")
                
                print(f"Epoch {epoch+1}: Val AUC = {auc_result:.4f}, Val ACC = {acc_metric:.4f}, Best AUC = {best_metric:.4f}")

    # 保存当前折的ROC数据（新增部分）
    fpr, tpr, _ = roc_curve(y_true_np, y_score)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_auc.append(auc(fpr, tpr))
    fold_results.append(best_metric)

# ==================== 新增：绘制五折ROC曲线 ====================
plt.figure(figsize=(10, 8))
mean_fpr = np.linspace(0, 1, 100)
tprs = []
best_aucs = []

# 计算每折插值后的TPR
for i in range(n_splits):
    interp_tpr = interp(mean_fpr, all_fpr[i], all_tpr[i])
    interp_tpr[0] = 0.0  # 确保从(0,0)开始
    tprs.append(interp_tpr)
    best_aucs.append(fold_results[i])

# 计算真正的平均AUC（基于每折AUC的平均）
mean_auc = np.mean(best_aucs)  # 修改点1：使用all_auc而非fold_results
std_auc = np.std(best_aucs)
ci95 = 1.96 * std_auc / np.sqrt(n_splits)  # 95%置信区间

# 计算平均TPR和置信区间
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0  # 确保结束于(1,1)
std_tpr = np.std(tprs, axis=0)

# 绘制每折曲线（使用all_auc而非fold_results）
for i in range(n_splits):
    plt.plot(all_fpr[i], all_tpr[i], alpha=0.3,
             label=f'Fold {i+1} (AUC = {best_aucs[i]:.4f})')  # 修改点2

# 绘制平均曲线和置信区间
plt.plot(mean_fpr, mean_tpr, color='b',
         label=f'Mean AUC = {mean_auc:.4f} (±{ci95:.4f})', lw=2)
plt.fill_between(mean_fpr, 
                 np.maximum(mean_tpr - std_tpr, 0),
                 np.minimum(mean_tpr + std_tpr, 1),
                 color='grey', alpha=0.2)


# 美化图形
plt.plot([0, 1], [0, 1], linestyle='--', color='r', label='Random Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('5-Fold Cross-Validation ROC Curves')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig('./crossval_pngs/crossval_multi2_tiaocan.png', dpi=300, bbox_inches='tight')
plt.show()

# 输出统计结果
print("\n=== Final Results ===")
print(f"Average AUC (mean of fold AUCs): {mean_auc:.4f} (±{ci95:.4f})")
print(f"Average Best Val AUC (clinical use): {np.mean(fold_results):.4f}")
print("Fold-wise AUCs:", [f"{x:.4f}" for x in all_auc])
print("Fold-wise Best Val AUCs:", [f"{x:.4f}" for x in fold_results])


In [ ]:
# 输出AUC、ACC、NPV、PPV等一系列指标的值和95%CI

def test_model_with_visualization(model, test_loader, device, num_classes=2, n_bootstrap=1000):
    """
    扩展的测试函数，包含：
    - 基础指标（AUC, ACC）及其95% CI
    - PPV/NPV/敏感性/特异性及其95% CI
    - 可视化功能
    """
    model.eval()
    y_pred_logits_list = [] # 存储模型的原始logits输出
    y_true_list = []
    test_loss_sum = 0
    loss_function = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(device))

    # 收集预测结果
    with torch.no_grad():
        for test_data in test_loader:
            pre_dce, pre_dwi, input_clinical, input_rad, val_labels = test_data["pre_dce"].to(device), test_data['pre_dwi'].to(device), test_data["clinical"].to(device), test_data["radiomics"].to(device), test_data["label"].to(device)
            outputs = model(pre_dce, pre_dwi, input_clinical, input_rad)
            y_pred_logits_list.append(outputs.cpu())
            y_true_list.append(val_labels.cpu())
            loss = loss_function(outputs, val_labels)
            test_loss_sum += loss.item() * val_labels.size(0) 
    
    y_pred_logits = torch.cat(y_pred_logits_list, dim=0) # [总样本数, num_classes]
    y_true = torch.cat(y_true_list, dim=0) # [总样本数]

    # 计算平均测试损失
    # 确保y_true不为空，避免除以零
    total_samples = len(y_true)
    test_loss_avg = test_loss_sum / total_samples if total_samples > 0 else 0.0

    # 将张量转换为numpy数组，并计算概率分数和预测标签
    y_true_np = y_true.numpy()
    # 将logits转换为概率，并取正类（通常为索引1）的概率
    y_prob_np = torch.softmax(y_pred_logits, dim=1).numpy() 
    y_prob_positive = y_prob_np[:, 1]  # 正类的概率
    # 根据阈值0.5获取预测的二分类标签
    y_pred_labels = (y_prob_positive > 0.5).astype(int) 

    # 基础指标点估计
    acc = accuracy_score(y_true_np, y_pred_labels)
    
    # 检查y_true_np是否包含至少两个类别，否则AUC和AP无法计算
    if len(np.unique(y_true_np)) < 2:
        roc_auc = np.nan # 设置为NaN，表示无法计算
        avg_precision = np.nan
        print("Warning: Only one class present in y_true_np. ROC AUC and Average Precision are not defined.")
    else:
        roc_auc = roc_auc_score(y_true_np, y_prob_positive)
        avg_precision = average_precision_score(y_true_np, y_prob_positive)

    # ============== Bootstrap置信区间 ==============
    def bootstrap_ci(metric_fn, y_true_data, y_pred_data, n_bootstrap=1000):
        """
        计算指标的95% Bootstrap置信区间。
        处理重采样子集只包含一个类别的情况。
        
        Args:
            metric_fn (function): 要计算的指标函数 (e.g., roc_auc_score, accuracy_score)。
            y_true_data (np.array): 真实标签数组。
            y_pred_data (np.array): 预测数据，可以是概率分数 (用于AUC) 或预测标签 (用于ACC)。
            n_bootstrap (int): Bootstrap重采样次数。
        Returns:
            tuple: (lower_bound, upper_bound) 95% 置信区间。
        """
        stats_list = []
        n_samples = len(y_true_data)
        
        if n_samples == 0: # 如果没有样本，返回NaN
            return (np.nan, np.nan)

        valid_samples_count = 0
        for _ in tqdm(range(n_bootstrap), desc=f"Bootstrap for {metric_fn.__name__}"):
            # 随机有放回地采样样本索引
            indices = np.random.choice(n_samples, n_samples, replace=True)
            
            resampled_y_true = y_true_data[indices]
            resampled_y_pred = y_pred_data[indices] # 可能是概率或标签

            # 对于ROC AUC和AP等需要多类别的指标，检查重采样数据
            if metric_fn in [roc_auc_score, average_precision_score]:
                if len(np.unique(resampled_y_true)) < 2:
                    # 如果重采样子集只包含一个类别，则跳过此次采样
                    continue 
            
            try:
                # 计算指标
                stat = metric_fn(resampled_y_true, resampled_y_pred)
                stats_list.append(stat)
                valid_samples_count += 1
            except ValueError as e:
                # 捕获可能的其他sklearn指标计算错误，并跳过
                print(f"Warning: Skipped a bootstrap sample due to error: {e}")
                continue # 跳过此样本

        # 如果有效采样数量不足，则置信区间可能不可靠
        if valid_samples_count < max(10, n_bootstrap * 0.01): # 至少10个有效采样，或者至少1%
             print(f"Warning: Only {valid_samples_count} valid bootstrap samples collected for {metric_fn.__name__}. CI may be unreliable.")
             return (np.nan, np.nan) 
             
        # 排序后取2.5%和97.5%分位数作为95%置信区间
        sorted_stats = np.sort(stats_list)
        lower = np.percentile(sorted_stats, 2.5)
        upper = np.percentile(sorted_stats, 97.5)
        return (lower, upper)

    # AUC和ACC的CI
    # AUC需要概率分数
    auc_ci = bootstrap_ci(roc_auc_score, y_true_np, y_prob_positive, n_bootstrap)
    # ACC需要预测标签
    acc_ci = bootstrap_ci(accuracy_score, y_true_np, y_pred_labels, n_bootstrap)
    
    # ============== 其他指标计算 ==============
    # 计算混淆矩阵 (True Negatives, False Positives, False Negatives, True Positives)
    tn, fp, fn, tp = confusion_matrix(y_true_np, y_pred_labels).ravel()
    
    # 点估计 (添加了分母为零的保护)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0 # 敏感性（召回率）：真阳性 / (真阳性 + 假阴性)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0 # 特异性：真阴性 / (真阴性 + 假阳性)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0         # 阳性预测值（Precision）：真阳性 / (真阳性 + 假阳性)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0         # 阴性预测值：真阴性 / (真阴性 + 假阴性)
    
    # 95%置信区间（Wilson Score Interval）函数
    def wilson_interval(pos, total):
        """计算比例的95% Wilson Score置信区间"""
        if total == 0: # 避免除以零
            return (0.0, 0.0)
        p = pos / total # 比例
        z = stats.norm.ppf(0.975) # 标准正态分布的97.5%分位数，对应95% CI
        
        # 计算区间边距，确保根号内的值非负
        sqrt_term = (p * (1 - p) + z**2 / (4 * total)) / total
        if sqrt_term < 0:
            sqrt_term = 0.0 # 理论上不会发生，但为数值稳定性考虑
        interval_margin = z * np.sqrt(sqrt_term)
        
        # Wilson Score Interval公式
        lower = (p + z**2 / (2 * total) - interval_margin) / (1 + z**2 / total)
        upper = (p + z**2 / (2 * total) + interval_margin) / (1 + z**2 / total)
        
        # 确保置信区间在[0, 1]范围内
        return (max(0.0, lower), min(1.0, upper))

    # 计算各项指标的Wilson置信区间
    sensitivity_ci = wilson_interval(tp, tp + fn)
    specificity_ci = wilson_interval(tn, tn + fp)
    ppv_ci = wilson_interval(tp, tp + fp)
    npv_ci = wilson_interval(tn, tn + fn)
    
    # ============== 可视化 ==============
    plt.figure(figsize=(15, 5))
    
    # 1. AUC-ROC曲线
    fpr, tpr, _ = roc_curve(y_true_np, y_prob_positive)
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC curve (AUC = {roc_auc:.4f}[{auc_ci[0]:.4f}, {auc_ci[1]:.4f}])')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")

    # 2. 添加PR曲线可视化（可选）
    plt.subplot(1, 2, 2)
    precision, recall, _ = precision_recall_curve(y_true_np, y_prob_positive)
    plt.plot(recall, precision, color='blue', lw=2,
             label=f'PR curve (AP = {avg_precision:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.show()

    # ============== 返回所有指标 ==============
    return {
        'y_true': y_true_np,
        'y_pred': y_prob_np,
        'y_pred_positive': y_prob_positive,  # 正类的概率
        'metrics': {
            'AUC': (roc_auc, auc_ci),
            'ACC': (acc, acc_ci),
            'SEN': (sensitivity, sensitivity_ci),
            'SPE': (specificity, specificity_ci),
            'PPV': (ppv, ppv_ci),
            'NPV': (npv, npv_ci),
            'Avg_Precision': avg_precision
        },
        'confusion_matrix': {
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn
        }
    }


In [ ]:
# 计算测试集的指标和预测分数
# 先准备：5折模型路径
folds = [1, 2, 3, 4, 5]
weights_path = './weights/first_weight/best_model_fold4.pth'
model_paths = [f"./weights_bigcrop/multi/best_model_fold{fold}.pth" for fold in folds]

# 创建一个目录来保存所有结果
output_dir = "./test_output"
os.makedirs(output_dir, exist_ok=True)

# 循环处理每家医院的测试数据
for hospital_id, test_data_for_hospital in test_data.items():
    if not test_data_for_hospital:
        print(f"\n--- No test data for Hospital {hospital_id}, skipping. ---")
        continue

    print(f"\n=============== Processing Hospital {hospital_id} ===============")

    # 创建当前医院的测试数据集和数据加载器
    test_ds_hospital = MyDataSet(patient_data_list=test_data_for_hospital, transform=val_preprocess)
    test_loader_hospital = DataLoader(test_ds_hospital, batch_size=16, num_workers=4,
                                       collate_fn=test_ds_hospital.collate_fn, pin_memory=True)

    # 准备结果存储（针对当前医院）
    hospital_all_metrics = []
    hospital_all_predictions = []

    # 循环5折
    for fold_id, model_path in enumerate(model_paths):
        print(f"\n=== Hospital {hospital_id} - Testing Fold {fold_id} ===")

        # 初始化模型
        model = DoubleTower(in_channel=1, dropout_rate=0.25, focal_alpha=[0.7, 0.3],focal_gamma=2.0, weights_path=weights_path,device=device)
        # 确保模型权重文件存在，否则会报错
        # 在实际运行中，你需要确保这些路径下的模型权重是可用的
        if not os.path.exists(model_path):
            print(f"Warning: Model weight not found at {model_path}. Skipping this fold for Hospital {hospital_id}.")
            # 可以选择在这里跳过当前fold，或者加载一个默认模型
            continue
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()

        # 测试
        results = test_model_with_visualization(
            model=model,
            test_loader=test_loader_hospital,
            device=device,
            num_classes=2,
            n_bootstrap=1000
        )

        metrics = results['metrics']

        # 打印
        print(f"[Hospital {hospital_id}, Fold {fold_id}] AUC: {metrics['AUC'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] ACC: {metrics['ACC'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] SEN: {metrics['SEN'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] SPE: {metrics['SPE'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] PPV: {metrics['PPV'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] NPV: {metrics['NPV'][0]:.4f}")
        print(f"[Hospital {hospital_id}, Fold {fold_id}] Avg Precision: {metrics['Avg_Precision']:.4f}")

        cm = results['confusion_matrix']
        print(f"[Hospital {hospital_id}, Fold {fold_id}] Confusion Matrix:")
        print(f"{'':<15}{'Predicted 1':<20}{'Predicted 0':<20}")
        print(f"{'Actual 1':<15}{cm['TP']:<20}{cm['FN']:<20}")
        print(f"{'Actual 0':<15}{cm['FP']:<20}{cm['TN']:<20}")

        # F1
        precision = metrics['PPV'][0]
        recall = metrics['SEN'][0]
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f"[Hospital {hospital_id}, Fold {fold_id}] F1 Score: {f1:.4f}")

        # 保存当前Fold指标
        hospital_all_metrics.append({
            'AUC': metrics['AUC'][0],
            'ACC': metrics['ACC'][0],
            'SEN': metrics['SEN'][0],
            'SPE': metrics['SPE'][0],
            'PPV': metrics['PPV'][0],
            'NPV': metrics['NPV'][0],
            'Avg_Precision': metrics['Avg_Precision'],
            'F1': f1
        })

        # 保存当前Fold的预测结果
        hospital_all_predictions.append({
            'fold': fold_id + 1,
            'patient_ids': [p['patient_id'] for p in test_data_for_hospital], # 添加 patient_id
            'y_true': results['y_true'],
            'y_pred': results['y_pred'],
            'y_pred_positive': results['y_pred_positive']
        })

    # =========== 计算并保存当前医院的5折平均指标 ============
    if hospital_all_metrics: # 只有当有数据时才计算平均值
        hospital_mean_metrics = {}
        for key in hospital_all_metrics[0].keys():
            hospital_mean_metrics[key] = np.mean([m[key] for m in hospital_all_metrics])

        # 打印平均指标
        print(f"\n=== Hospital {hospital_id} - 5折平均指标 ===")
        for key, value in hospital_mean_metrics.items():
            print(f"{key}: {value:.4f}")

        # 保存CSV
        csv_filename = os.path.join(output_dir, f"hospital_{hospital_id}_metrics_multi.csv")
        with open(csv_filename, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)

            # 表头
            header = ['指标']
            for i in range(len(hospital_all_metrics)):
                header.append(f'Fold{i+1}')
            header.append('平均')
            writer.writerow(header)

            # 每个指标写一行
            for key in hospital_mean_metrics.keys():
                row = [key]
                for m in hospital_all_metrics:
                    row.append(m[key])
                row.append(hospital_mean_metrics[key])
                writer.writerow(row)

        print(f"\nHospital {hospital_id} 指标已保存到 {csv_filename}")

        # =========== 保存当前医院的预测概率CSV ============
        predictions_csv_filename = os.path.join(output_dir, f"hospital_{hospital_id}_pred_multi.csv")
        with open(predictions_csv_filename, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)

            # 表头
            header = ['Patient_ID', 'True_Label']
            for i in range(len(hospital_all_predictions)):
                header.extend([f'Fold{i+1}_Prob_0', f'Fold{i+1}_Prob_1', f'Fold{i+1}_Pred_Prob'])
            header.extend(['Avg_Prob_0', 'Avg_Prob_1', 'Avg_Pred_Prob'])
            writer.writerow(header)

            # 获取样本数量 (假设所有fold的样本数量一致)
            num_samples = len(hospital_all_predictions[0]['y_true'])

            # 计算平均概率
            # 过滤掉空的预测结果，以防某些fold因为模型权重缺失而跳过
            valid_predictions = [pred for pred in hospital_all_predictions if pred['y_pred'].shape[0] == num_samples]
            if valid_predictions:
                avg_prob_0 = np.mean([pred['y_pred'][:, 0] for pred in valid_predictions], axis=0)
                avg_prob_1 = np.mean([pred['y_pred'][:, 1] for pred in valid_predictions], axis=0)
                avg_pred_prob = np.mean([pred['y_pred_positive'] for pred in valid_predictions], axis=0)

                # 为每个样本写入一行
                for i in range(num_samples):
                    row = [
                        hospital_all_predictions[0]['patient_ids'][i], # 患者ID
                        hospital_all_predictions[0]['y_true'][i],     # 真实标签（所有fold相同）
                    ]

                    # 添加每个fold的概率
                    for pred in hospital_all_predictions:
                        # 确保当前fold的预测结果不为空且与样本数量匹配
                        if pred['y_pred'].shape[0] == num_samples:
                            row.extend([
                                pred['y_pred'][i, 0],   # 类别0的概率
                                pred['y_pred'][i, 1],   # 类别1的概率
                                pred['y_pred_positive'][i] # 预测为正类的概率
                            ])
                        else:
                            # 如果某个fold的数据不完整，用空值填充
                            row.extend(['', '', ''])

                    # 添加平均概率
                    row.extend([
                        avg_prob_0[i],
                        avg_prob_1[i],
                        avg_pred_prob[i]
                    ])

                    writer.writerow(row)
                print(f"\nHospital {hospital_id} 预测概率已保存到 {predictions_csv_filename}")
            else:
                print(f"No valid predictions found for Hospital {hospital_id} to calculate average probabilities.")
    else:
        print(f"No valid metrics calculated for Hospital {hospital_id}.")

print("\nAll hospital-specific testing and saving completed.")
